# CoverFind — Data Pre-Processing
Steps: install packages → load data → drop unnecessary columns

## Step 1 — Install & Import Packages

In [ ]:
# Install required packages (run once)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas', 'openpyxl', 'numpy', '-q'])
print('All packages installed.')

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

DATASET_DIR = r'C:\Users\harip\Documents\Hackathon\dataset'
INSURANCE_DIR = DATASET_DIR + r'\Insurance'

print('Packages loaded.')

## Step 2 — Load Data

### 2a. HRSA Health Centers (providers)

In [ ]:
hrsa_raw = pd.read_excel(
    DATASET_DIR + r'\Health_Center_Service_Delivery_and_LookAlike_Sites.xlsx'
)
print(f'HRSA shape: {hrsa_raw.shape}')
hrsa_raw.head(3)

### 2b. Insurance — Plan Attributes

In [ ]:
plan_attrs_raw = pd.read_csv(INSURANCE_DIR + r'\PlanAttributes.csv', low_memory=False)
print(f'PlanAttributes shape: {plan_attrs_raw.shape}')
plan_attrs_raw.head(3)

### 2c. Insurance — Rates

In [ ]:
rates_raw = pd.read_csv(INSURANCE_DIR + r'\Rate.csv', low_memory=False)
print(f'Rate shape: {rates_raw.shape}')
rates_raw.head(3)

### 2d. Insurance — Service Areas

In [ ]:
service_area_raw = pd.read_csv(INSURANCE_DIR + r'\ServiceArea.csv', low_memory=False)
print(f'ServiceArea shape: {service_area_raw.shape}')
service_area_raw.head(3)

### 2e. Insurance — Networks

In [ ]:
network_raw = pd.read_csv(INSURANCE_DIR + r'\Network.csv', low_memory=False)
print(f'Network shape: {network_raw.shape}')
network_raw.head(3)

## Step 3 — Drop Unnecessary Columns

### 3a. HRSA — Keep only location, identity, type & coordinates

In [ ]:
HRSA_KEEP = [
    'Health Center Number',                             # unique provider ID
    'FQHC Site NPI Number',                            # NPI for cross-referencing
    'Site Name',                                        # clinic name
    'Site Address',                                     # street address
    'Site City',
    'Site State Abbreviation',
    'Site Postal Code',
    'Site Telephone Number',
    'Site Web Address',
    'Health Center Service Delivery Site Location Setting Description',  # urban/rural etc.
    'Site Status Description',                          # active / inactive
    'Health Center Type Description',                   # FQHC / look-alike etc.
    'Health Center Name',                               # parent org name
    'Geocoding Artifact Address Primary X Coordinate',  # longitude
    'Geocoding Artifact Address Primary Y Coordinate',  # latitude
    'State Name',
    'Complete County Name',
    'HHS Region Name',
]

hrsa = hrsa_raw[HRSA_KEEP].copy()

# Rename for clarity
hrsa.rename(columns={
    'Health Center Number': 'provider_id',
    'FQHC Site NPI Number': 'npi',
    'Site Name': 'site_name',
    'Site Address': 'address',
    'Site City': 'city',
    'Site State Abbreviation': 'state',
    'Site Postal Code': 'zip',
    'Site Telephone Number': 'phone',
    'Site Web Address': 'website',
    'Health Center Service Delivery Site Location Setting Description': 'location_setting',
    'Site Status Description': 'status',
    'Health Center Type Description': 'center_type',
    'Health Center Name': 'org_name',
    'Geocoding Artifact Address Primary X Coordinate': 'longitude',
    'Geocoding Artifact Address Primary Y Coordinate': 'latitude',
    'State Name': 'state_name',
    'Complete County Name': 'county',
    'HHS Region Name': 'hhs_region',
}, inplace=True)

print(f'HRSA after column drop: {hrsa.shape}')
hrsa.head(3)

### 3b. Plan Attributes — Keep plan identity, metal level, type, and cost-sharing

In [ ]:
PLAN_KEEP = [
    'BusinessYear',
    'StateCode',
    'IssuerId',
    'PlanId',
    'PlanMarketingName',
    'PlanType',           # HMO, PPO, EPO, POS
    'MetalLevel',         # Bronze, Silver, Gold, Platinum
    'MarketCoverage',     # Individual / SHOP
    'DentalOnlyPlan',     # filter these out later
    # In-network Tier 1 deductible (individual) — core cost signal
    'DEHBDedInnTier1Individual',
    # In-network Tier 1 out-of-pocket max (individual)
    'DEHBInnTier1IndividualMOOP',
    'ChildOnlyOffering',
]

plan_attrs = plan_attrs_raw[PLAN_KEEP].copy()

plan_attrs.rename(columns={
    'BusinessYear': 'year',
    'StateCode': 'state',
    'IssuerId': 'issuer_id',
    'PlanId': 'plan_id',
    'PlanMarketingName': 'plan_name',
    'PlanType': 'plan_type',
    'MetalLevel': 'metal_level',
    'MarketCoverage': 'market',
    'DentalOnlyPlan': 'dental_only',
    'DEHBDedInnTier1Individual': 'deductible_individual',
    'DEHBInnTier1IndividualMOOP': 'oop_max_individual',
    'ChildOnlyOffering': 'child_only',
}, inplace=True)

print(f'PlanAttributes after column drop: {plan_attrs.shape}')
plan_attrs.head(3)

### 3c. Rates — Keep plan key + individual premium

In [ ]:
RATE_KEEP = [
    'BusinessYear',
    'StateCode',
    'IssuerId',
    'PlanId',
    'RatingAreaId',
    'Age',
    'IndividualRate',
]

rates = rates_raw[RATE_KEEP].copy()

rates.rename(columns={
    'BusinessYear': 'year',
    'StateCode': 'state',
    'IssuerId': 'issuer_id',
    'PlanId': 'plan_id',
    'RatingAreaId': 'rating_area',
    'Age': 'age',
    'IndividualRate': 'monthly_premium',
}, inplace=True)

print(f'Rates after column drop: {rates.shape}')
rates.head(3)

### 3d. Service Area — Keep geographic coverage keys

In [ ]:
SA_KEEP = [
    'BusinessYear',
    'StateCode',
    'IssuerId',
    'ServiceAreaId',
    'ServiceAreaName',
    'CoverEntireState',
    'County',
    'ZipCodes',
    'MarketCoverage',
]

service_area = service_area_raw[SA_KEEP].copy()

service_area.rename(columns={
    'BusinessYear': 'year',
    'StateCode': 'state',
    'IssuerId': 'issuer_id',
    'ServiceAreaId': 'service_area_id',
    'ServiceAreaName': 'service_area_name',
    'CoverEntireState': 'covers_entire_state',
    'County': 'county',
    'ZipCodes': 'zip_codes',
    'MarketCoverage': 'market',
}, inplace=True)

print(f'ServiceArea after column drop: {service_area.shape}')
service_area.head(3)

### 3e. Network — Keep network identity

In [ ]:
NET_KEEP = [
    'BusinessYear',
    'StateCode',
    'IssuerId',
    'NetworkId',
    'NetworkName',
    'MarketCoverage',
]

network = network_raw[NET_KEEP].copy()

network.rename(columns={
    'BusinessYear': 'year',
    'StateCode': 'state',
    'IssuerId': 'issuer_id',
    'NetworkId': 'network_id',
    'NetworkName': 'network_name',
    'MarketCoverage': 'market',
}, inplace=True)

print(f'Network after column drop: {network.shape}')
network.head(3)

## Quick Sanity Check

In [ ]:
print('=== HRSA ===')
print(hrsa.dtypes)
print(f'\nActive sites: {(hrsa.status == "Active").sum()} / {len(hrsa)}')
print(f'States covered: {hrsa.state.nunique()}')

print('\n=== Plans ===')
print(plan_attrs.metal_level.value_counts())
print(plan_attrs.plan_type.value_counts())

print('\n=== Rates sample ===')
print(rates.monthly_premium.describe())